In [6]:
from ngsolve import *
import numpy as np
import scipy.optimize

class gel_3D:

    def __init__(
        self,
        length=90.0,
        width=15.0,
        thickness=1.6,
        phi0=0.2,
        mu_bar=-0.03
    ):

        self.phi0 = phi0

        # -------------------------------------------------
        # thermodynamic constants
        # -------------------------------------------------

        T = 25 + 273.15
        K_B = 1.380649e-23
        V_m = 3e-29

        self.entropic_unit = K_B*T/V_m*1e-6

        self.gamma = 0.001
        self.chi = 0.4

        self.G = self.gamma*self.entropic_unit

        vapor_pressure = 3.2e-3

        self.p0_bar = vapor_pressure/self.entropic_unit

        # use NGSolve Parameters
        self.mu_bar = Parameter(mu_bar)

        self.p_bar = Parameter(
            self.p0_bar*np.exp(mu_bar)
        )

        # geometry
        self.L = length
        self.w = width
        self.d = thickness

        print(f'phi0 = {self.phi0}')
        print(f'mu_bar = {mu_bar}')
        print(f'entropic_unit = {self.entropic_unit}')
        print(f'gamma = {self.gamma}')
        print(f'chi = {self.chi}')

    # =====================================================
    # NUMPY FUNCTIONS
    # =====================================================

    def phi_numpy(self, J):

        return self.phi0/J

    def dH_numpy(self, J):

        if J <= self.phi0:
            return 1e20

        phi = self.phi_numpy(J)

        mu = float(self.mu_bar.Get())

        p = float(self.p_bar.Get())

        return (
            phi
            + np.log(1-phi)
            + self.chi*phi**2
            - self.gamma/J
            + p
            - mu
        )

    # =====================================================
    # NGSOLVE FUNCTIONS
    # =====================================================

    def phi(self, J):

        return self.phi0/J

    def H(self, J):

        phi = self.phi(J)

        eps = 1e-12

        # safe log(1-phi)

        one_minus_phi_safe = IfPos(
            1 - phi - eps,
            1 - phi,
            eps
        )

        # safe log(J)

        J_safe = IfPos(
            J - eps,
            J,
            eps
        )

        return (
            (J-self.phi0)*log(one_minus_phi_safe)
            + self.phi0*self.chi*(1-phi)
            - self.gamma*log(J_safe)
            + (self.p_bar-self.mu_bar)*(J-self.phi0)
        )

    # =====================================================
    # ENERGY
    # =====================================================

    def W(self, F):

        J = Det(F)

        C = F.trans * F

        return (
            0.5*self.G*(Trace(C)-3)
            + self.entropic_unit*self.H(J)
        )

    # =====================================================
    # COMPUTE MU FROM LAMBDA
    # =====================================================

    def mu_fun(self, lamb):

        phi0 = self.phi0
        gamma = self.gamma
        chi = self.chi
        p0_bar = self.p0_bar

        def residual(mu_bar):

            p_bar = p0_bar*np.exp(mu_bar)

            phi = phi0/lamb

            if phi >= 1:
                return 1e20

            return (
                phi
                + np.log(1-phi)
                + chi*phi**2
                - gamma/lamb
                + p_bar
                - mu_bar
            )

        mu_guess = -0.03

        mu_sol = scipy.optimize.fsolve(
            residual,
            mu_guess
        )[0]

        print(f'lambda = {lamb}, computed mu = {mu_sol}')

        return mu_sol

In [10]:
from ngsolve.webgui import Draw
### Main ###
L = 90.0
d = 1.62
w = 15.0

phi0 = 0.20
muBarAbs = 0.001000
mu_bar = - muBarAbs
#
# indexes_iter = [10]
iter_first = 0
iter_last = 10
indexes_iter = list(range(iter_first, iter_last+1))
#
order = 2
mesh_file = 'meshes/mesh0.vol.gz'

print(f'L={L}; w={w}; d={d}; phi0={phi0}; mu_bar={mu_bar}; order={order}')
print(f'indexes_iter={indexes_iter}')
print(f'mesh_file = {mesh_file}')

# Creates the 'gel'
gel = gel_3D(length=L,width=w,thickness=d,phi0=phi0,mu_bar=mu_bar)
# Loads the mesh
mesh = Mesh(mesh_file)
# Creates the finite element space
fes = VectorH1(mesh, order=order, dirichlet="bonded|debonded")
print('nDoF = {}'.format(fes.ndof))

# Creates the auxiliary strings filename_suffix and filename to be used later to construct the filename_iter
filename_suffix = (
    f'_phi0={phi0:.2f}'
    f'_muBarAbs={np.abs(mu_bar):.6f}'
)
filename = (
    'result'
    + filename_suffix
    + f'_order={order}'
)

for numIteration in indexes_iter:
    # Constructs the string filename_iter
    filename_iter = (
        filename
        + "_iter="
        + str(numIteration).zfill(2)
    )

    # Creates an object of type gridfunction, where we will load the previously computed solution of the equations
    gfu = GridFunction(fes)
    # Loads the .gfu
    filename_gridfunction = 'gridfunctions/' + filename_iter + '.gfu'
    gfu.Load(filename_gridfunction)
    I = Id(mesh.dim)
    F= I + Grad(gfu)

    # computes energy density as H1 function (takes the projection to H1 of W(F), which is only in L2)
    GF_energy_density = GridFunction(H1(mesh, order=1))
    GF_energy_density.Set(gel.W(F))
    # Draw(GF_energy_density*1e3, mesh, deformation=gfu, min =0.0, max = 152.0)        # Energy densities in [KPa]
    Draw(GF_energy_density*1e3, mesh, deformation=gfu, min =0.0)        # Energy densities in [KPa]
    # Compute the total energy
    elasticEnergy = Integrate(GF_energy_density, mesh, order=5)
    print('Total energy [mJ]: {:.2f}'.format(elasticEnergy))

    filename_vtk = 'scratchpad.nosync/' + filename_iter
    vtk = VTKOutput(mesh,coefs=[gfu, GF_energy_density],names=["Displacement", "Energy density"],filename=filename_vtk,subdivision=1)
    vtk.Do()








L=90.0; w=15.0; d=1.62; phi0=0.2; mu_bar=-0.001; order=2
indexes_iter=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
mesh_file = meshes/mesh0.vol.gz
phi0 = 0.2
mu_bar = -0.001
entropic_unit = 137.21349978333333
gamma = 0.001
chi = 0.4
nDoF = 586314


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

Total energy [mJ]: -34143.86


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

Total energy [mJ]: -34062.63


...

Total energy [mJ]: -33573.73


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

Total energy [mJ]: -33549.39


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

Total energy [mJ]: -33518.15
